In [ ]:
import os
import requests
import json
from pathlib import Path
import datetime

ACCESS_KEY = ""
HEADERS = {"Authorization": f"Client-ID {ACCESS_KEY}"}

# Dossiers
IMAGES_DIR = Path("images")
DATA_DIR = Path("data")
IMAGES_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

CATEGORIES = {
    "animaux": "animals",
    "batiments": "architecture",
    "paysage": "landscape",
}

metadata_list = []

for category_name, query in CATEGORIES.items():
    print(f"Chargement de la catégorie {category_name}...")

    url = "https://api.unsplash.com/search/photos"
    params = {
        "query": query,
        "per_page": 33,
        "page": 1,
    }

    response = requests.get(url, headers=HEADERS, params=params, timeout=30)
    response.raise_for_status()
    results = response.json()["results"]

    for i, photo in enumerate(results[:33], start=1):
        photo_id = photo["id"]
        filename = f"{category_name}_{photo_id}.jpg"

        # URL de l’image (format “regular” utilisé ici)
        image_url = photo["urls"]["regular"]

        # Téléchargement binaire de l’image
        image_response = requests.get(image_url, stream=True, timeout=30)
        image_response.raise_for_status()

        filepath = IMAGES_DIR / filename

        # Sauvegarde de l’image
        with open(filepath, "wb") as f:
            for chunk in image_response.iter_content(chunk_size=8192):
                f.write(chunk)

        # Calcul taille fichier Ko
        file_stat = filepath.stat()
        file_size_kb = round(file_stat.st_size / 1024, 2)

        # Metadonnées EXIF si disponibles (Unsplash fournit ces champs dans l’API)
        exif = photo.get("exif", {})
        camera = exif.get("make", "") + " " + exif.get("model", "")
        camera = camera.strip()
        aperture = exif.get("aperture")
        focal_length = exif.get("focal_length")
        iso = exif.get("iso")
        exposure = exif.get("exposure_time")

        # Date de prise de vue (exif.created_at si disponible, sinon created_at du photo)
        taken_at = exif.get("created_at", "") or photo.get("created_at", "")

        # Création de l’entrée métadata
        meta = {
            "nom_fichier": filename,
            "categorie": category_name,
            "largeur": photo["width"],
            "hauteur": photo["height"],
            "format_fichier": "jpg",
            "taille_fichier_kb": file_size_kb,
            "url_source": photo["urls"]["full"],
            "licence": "MIT",
            "description": photo.get("description") or photo.get("alt_description"),
            "photographe": photo["user"]["name"],
            "page_unsplash": photo["links"]["html"],
            "date_creation_photo": photo["created_at"],
            "nombre_likes": photo.get("likes", 0),
            "exif": {
                "camera_model": camera if camera else None,
                "taken_at": taken_at if taken_at else None,
                "aperture": aperture,
                "focal_length": focal_length,
                "iso": iso,
                "exposure_time": exposure,
            },
        }

        metadata_list.append(meta)

# Sauvegarde des métadonnées dans data/images_metadata.json
output_path = DATA_DIR / "images_metadata.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(metadata_list, f, ensure_ascii=False, indent=2)

print(f"✅ {len(metadata_list)} images et métadonnées sauvegardées.")

Chargement de la catégorie animaux...
Chargement de la catégorie batiments...
Chargement de la catégorie paysage...
✅ 30 images et métadonnées sauvegardées.
